# 1.2 Option A (Default): First Image Through an Endpoint

This notebook runs on your **laptop CPU** and calls a GPU model deployed as a Nebius Serverless AI Endpoint from the [Serverless AI Cookbook](https://github.com/nebius/serverless-ai-cookbook/blob/main/templates/endpoint-sana/README.md).

We adapt the `ImageGenerator` pattern and Emma's holiday gift-card scenario from the [Build a Media Agent workshop](https://github.com/DataTalksClub/build-media-agent-workshop), using **Sana 1.6B on L40S** as the course default.

In [ ]:
import json
import os
from pathlib import Path
import sys

MODULE = "01-consistent-media-generation-on-serverless-gpus"
candidates = [Path.cwd(), Path.cwd() / "code", Path.cwd() / MODULE / "code"]
CODE = next(p for p in candidates if (p / "generation.json").is_file())
sys.path.insert(0, str(CODE.resolve()))

BASE_URL = os.environ.get("BASE_URL")
TOKEN = os.environ.get("TOKEN")
print("Code directory:", CODE.resolve())
print("BASE_URL configured:", bool(BASE_URL), "| TOKEN configured:", bool(TOKEN))

## 1. Creative Brief and Repeatable Configuration

In our Personalized Media Factory, each record describes a recipient and their wish. In Module 2 an LLM agent will write the prompt dynamically; here in Module 1 we keep the prompt, model, seed, and parameters fixed in `generation.json` so we can measure the GPU worker on a repeatable input.

In [ ]:
from common import config

kid = {"name": "Emma", "age": 7, "wishlist": ["a telescope"]}
cfg = config(CODE / "generation.json")
print("Recipient record:", kid)
print(json.dumps(cfg, indent=2))

## 2. Wait for Readiness (`RUNNING` != Ready)

When the console shows `RUNNING`, the container has started, but the server may still be downloading and loading model weights into GPU memory. During that window, `GET /v1/models` can return `502 Bad Gateway`. `wait_ready()` polls `/v1/models` until it returns HTTP `200` with a non-empty model list.

In [ ]:
from call_endpoint import wait_ready

models = wait_ready(BASE_URL, TOKEN)
print("Endpoint ready. Served models:", [m.get("id") for m in models["data"]])

## 3. Generate One Image with `ImageGenerator`

`ImageGenerator` wraps `POST /v1/images/generations`, decodes the base64 PNG response, and saves the file locally.

In [ ]:
from IPython.display import Image, display
from call_endpoint import ImageGenerator

sana_generator = ImageGenerator(BASE_URL, TOKEN, cfg["model"])
png = sana_generator.generate_image(
    cfg["prompt"],
    seed=cfg["seed"],
    size=f'{cfg["params"]["width"]}x{cfg["params"]["height"]}',
    num_inference_steps=cfg["params"]["num_inference_steps"],
    guidance_scale=cfg["params"]["guidance_scale"],
)
display(Image(data=png))

## 4. Run 3 Independent Generations and Record `metrics.json`

For Homework 1 we generate the asset 3 times with the same configuration to record independent request latencies (`latency_s`), SHA-256 hashes, and trace events (`trace.jsonl`).

In [ ]:
from call_endpoint import run

folder = run(CODE / "generation.json", n=3)
report = json.loads((folder / "metrics.json").read_text())
print(json.dumps(report, indent=2))

## 5. Compute Cost per Asset and API Breakeven

In Lesson 1.5 we turn startup, readiness wait, and generation latency into marginal cost `c`, fixed cost `F`, and total cost per asset using a simple Python function `calculate_cost`.

In [ ]:
from common import calculate_cost

# Replace startup_s, ready_s, and price_per_hour with your observed values if different:
cost_summary = calculate_cost(
    startup_s=report.get("startup_s") or 180.0,
    ready_s=report.get("ready_wait_s") or 40.0,
    latency_s=report["latency_s"],
    price_per_hour=1.5484,  # L40S 1gpu-8vcpu-32gb on-demand rate ($/h)
    counts=(1, 10, 100),
    api_price=0.04,         # Example per-call image API price ($/image)
)
print(json.dumps(cost_summary, indent=2))

## 6. Optional: Compare with `Qwen/Qwen-Image`

If you also deployed the `Qwen-Image` template from the cookbook and set `QWEN_ENDPOINT_URL` and `QWEN_ENDPOINT_TOKEN`, run the same prompt and seed to compare visual style and latency.

In [ ]:
qwen_url = os.environ.get("QWEN_ENDPOINT_URL")
qwen_token = os.environ.get("QWEN_ENDPOINT_TOKEN")

if qwen_url and qwen_token:
    qwen_generator = ImageGenerator(qwen_url, qwen_token, "Qwen/Qwen-Image")
    qwen_png = qwen_generator.generate_image(cfg["prompt"], seed=cfg["seed"])
    display(Image(data=qwen_png))
else:
    print("Skipping optional Qwen-Image comparison (QWEN_ENDPOINT_URL not set).")

## 7. Stop or Delete the Endpoint

An Endpoint bills per second while it is `RUNNING`, even when no requests are sent. Stop or delete your Endpoint in the Nebius console now and verify that it is no longer `RUNNING`.

In [ ]:
print("ACTION REQUIRED: Stop or delete the Endpoint in the Nebius console and confirm it is no longer RUNNING.")